# 01 — Radial integration from a detector image

End-to-end CPU pipeline against the bundled CeO₂ Pilatus frame:

1. Build a pixel → (R, η) map via `MIDASDetectorMapper`.
2. Pack the TIFF into a MIDAS zarr.zip.
3. Integrate via `MIDASIntegrator`.
4. Load the 2D cake + plot a 1D lineout.

No external data needed — the wheel ships everything.

In [1]:
import shutil, tempfile
from pathlib import Path

import midas_auto_calibrate as mac
import midas_integrate as mi

print('midas-integrate', mi.__version__)
print('midas-auto-calibrate', mac.__version__)
print('shipped MIDASIntegrator:', mi.midas_bin('MIDASIntegrator'))

midas-integrate 0.1.0
midas-auto-calibrate 0.1.0
shipped MIDASIntegrator: /Users/hsharma/miniconda3/envs/midas_env/lib/python3.12/site-packages/midas_integrate/_bin/MIDASIntegrator


In [2]:
workdir = Path(tempfile.mkdtemp(prefix='mi_nb_'))
img = workdir / 'CeO2_00001.tif'
shutil.copy(mac.data.CEO2_PILATUS, img)

cfg = mi.IntegrationConfig(
    lsd=657_436.9, ybc=685.5, zbc=921.0,
    wavelength=0.172973, pixel_size=172.0,
    nr_pixels_y=1475, nr_pixels_z=1679,
    r_min=50, r_max=1200, r_bin_size=0.5,
    eta_min=-180, eta_max=180, eta_bin_size=1.0,
)

## Step 1 — Build the pixel → (R, η) map

`Mapper` pre-computes the per-pixel binning weights using Green's theorem.
This is reusable across many frames — build once per geometry.

In [3]:
artefacts = mi.Mapper(cfg).build(workdir, n_cpus=4)
print(f'Map.bin   : {artefacts.map_bin.stat().st_size / 1e6:.1f} MB')
print(f'nMap.bin  : {artefacts.n_map_bin.stat().st_size / 1e3:.1f} KB')

Map.bin   : 234.9 MB
nMap.bin  : 6624.1 KB


## Step 2 — Pack the TIFF into a MIDAS zarr.zip

`IntegratorZarrOMP` expects its input frame inside a zarr.zip with a specific
schema (see ``midas_integrate/io.py`` for the layout).

In [4]:
zarr_zip = workdir / 'CeO2.zarr.zip'
mi.make_zarr_zip(img, cfg, zarr_zip)
print(f'{zarr_zip.name}  ({zarr_zip.stat().st_size / 1e6:.1f} MB)')

CeO2.zarr.zip  (3.4 MB)


## Step 3 — Integrate

In [5]:
result = mi.Integrator(cfg, artefacts).integrate(zarr_zip, n_cpus=4)
print(f'cake: {result.cake_path.name}')
cake = result.load_cake()
print(f'  R bins : {len(cake["R"])}')
print(f'  η bins : {len(cake["Eta"])}')
print(f'  I shape: {cake["I"].shape}')
print(f'  max I  : {cake["I"].max():.1f}')

cake: CeO2.zarr.zip.caked.hdf
  R bins : 2300
  η bins : 360
  I shape: (2300, 360)
  max I  : 13764.6


## Step 4 — Plot the 1D lineout

Average over η → diffraction rings show as peaks in R.

In [6]:
try:
    import matplotlib.pyplot as plt
    I_1d = cake['I'].mean(axis=1)   # (nR,) = mean over η
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(cake['R'], I_1d, linewidth=0.8)
    ax.set_xlabel('R (pixels)')
    ax.set_ylabel('Mean intensity')
    ax.set_title('CeO₂ Pilatus — 1D lineout')
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    plt.show()
except ImportError:
    print('matplotlib not installed — skipping plot')

/var/folders/qw/k6gzh2ws7w397493kq4vnl_w0001pb/T/ipykernel_92624/853147133.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
